# Model Tester

In [17]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Setup

In [18]:
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"
# os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.losses import MeanSquaredError, mse
from tensorflow.keras.callbacks import (
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, LearningRateSchedule

from scripts.core import Core
from scripts.utils import setup_logging, load_data
from scripts.utils.plots import plot_predictions, plot_histogram
from scripts.utils.tf.dataloaders import *
from scripts.utils.tf.plots import plot_metrics
from scripts.utils.tf.callbacks import TimedLoggingCallback, WarmupLearningRate
from scripts.trainer import *


In [19]:
logger = setup_logging(__name__, level=logging.INFO)
logging.getLogger("scripts").setLevel(logging.DEBUG)
logging.getLogger("tensorflow").setLevel(logging.ERROR)


In [20]:
class CustomSchedule(LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.d_model = d_model
        self.d_model = tf.cast(self.d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps**-1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)


## Configure

In [21]:
# s = Core(["settings/planck.json", "--no-lensing", "--no-noise", "--narray", "500"])
s = Core(["settings/planck.json"])

MAX_EPOCHS = 100
BATCH_SIZE = 8

# just some info for the model name
timestamp = int(time.time())
model_settings = {
    # "dropout_rate": 0.1,
    "name": f"tester-{s.base_name}-{timestamp}",
}

data_loader_args = {
    "shuffle": True,
    "shuffle_buffer": 1000,
    "seed": None,
    "batch_size": BATCH_SIZE,
    "cache": True,
    "normalize": True,
}

# additional metrics we are interested in
metrics = ["mean_absolute_error"]

callbacks = [
    # EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    # TimedLoggingCallback(print_frequency=3),
    # TensorBoard(log_dir=f"{s.tb_dir}/{model_settings['name']}"),
    TerminateOnNaN(),
]


18-Apr-24 12:44:10 - scripts.core - DEBUG - Parsing CLI args: ['settings/planck.json']

18-Apr-24 12:44:10 - scripts.core - INFO - Loading settings from file settings/planck.json

18-Apr-24 12:44:10 - scripts.core - DEBUG - Found non-default value for 'cosmo_params': {'H0': 70.1, 'As':         
2.457e-09, 'ns': 0.96, 'ombh2': 0.02256, 'omch2': 0.1143, 'tau': 0.084, 'max_l': 1500, 'lmax': 1024} (default: {})

18-Apr-24 12:44:10 - scripts.core - DEBUG - Overriding cosmo param As from 2.13e-09 to 2.457e-09

18-Apr-24 12:44:10 - scripts.core - DEBUG - Overriding cosmo param ns from 0.9624 to 0.96

...

18-Apr-24 12:44:12 - scripts.core - DEBUG - Setting 'tb_dir' not found, using default: 'tensorboard'

18-Apr-24 12:44:12 - scripts.core - DEBUG - Setting 'model_dir' not found, using default: 'models'

18-Apr-24 12:44:12 - scripts.core - DEBUG - Setting 'alm_dir' not found, using default: 'alms'

18-Apr-24 12:44:12 - scripts.core - DEBUG - Setting 'patch_dir' not found, using default: 'patches'

AttributeError: 'Core' object has no attribute 'npatches'

In [ ]:
# import wandb
# from wandb.keras import WandbMetricsLogger

# # wandb.tensorboard.patch(root_logdir=s.tb_dir)

# wandb.init(
#     project="mlpng",
#     tags=["attn-alm", "dev"],
#     config=s.settings | model_settings,
#     dir="data",
#     sync_tensorboard=True,
# )

# callbacks.append(WandbMetricsLogger())


## Model 1

In [ ]:
data_loader = AlmLoader(
    s.alm_file,
    channels_last=True,
    **data_loader_args,
).as_tfds(auto_convert=True)

train_dataset, test_dataset, val_dataset = data_loader.get_split(0.8, 0.1, 0.1)
learning_rate = CustomSchedule(1024)
# learning_rate = WarmupLearningRate()
# learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model = alm_modelV1_2(Input(data_loader.shape), **model_settings)  # type: ignore
    model.compile(optimizer=opt, loss=mse, metrics=metrics)

model.summary()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

try:
    with h5py.File(s.alm_file, "r", swmr=True, locking=False) as hdf:
        fisher = hdf.get("fisher", [None])[0]
        logger.info(f"Loaded fisher matrix: {fisher}")
except Exception as e:
    logger.error(f"Could not load fisher matrix: {e}")
    fisher = None

y_pred = model.predict(test_dataset, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_dataset])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)


## Patch Model

In [ ]:
data_loader = PatchLoader(
    s.patch_file,
    # channels_last=True,
    **data_loader_args,
).as_tfds(auto_convert=True)

train_dataset, test_dataset, val_dataset = data_loader.get_split(0.8, 0.1, 0.1)
learning_rate = CustomSchedule(1024)
# learning_rate = WarmupLearningRate()
# learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model = patch_modelV1(Input(data_loader.shape), **model_settings)  # type: ignore
    model.compile(optimizer=opt, loss=mse, metrics=metrics)

model.summary()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

try:
    with h5py.File(s.alm_file, "r", swmr=True, locking=False) as hdf:
        fisher = hdf.get("fisher", [None])[0]
        logger.info(f"Loaded fisher matrix: {fisher}")
except Exception as e:
    logger.error(f"Could not load fisher matrix: {e}")
    fisher = None

y_pred = model.predict(test_dataset, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_dataset])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)
